## 005

In [2]:
# set으로 문서 키워드 집합 정의
set_A = {'자연어', '처리', '머신', '러닝', '알고리즘'}
set_B = {'인공지능', '머신', '러닝', '딥러닝', '알고리즘'}

# set 내장 메서드로 교집합, 합집합 계산
# intersection() : 두 집합의 공통 원소만 반환
# union() : 두 집합의 모든 원소 반환 (중복 제거)
intersection = set_A.intersection(set_B)
union = set_A.union(set_B)
# intersection()과 union() 은 파이썬 set의 내장 메서드

# 자카드 = 교집합 / 합집합
jacard_sim = len(intersection) / len(union)

# 다이스 = 2 X 교집합 / (|A| + |B|)
dice_sim = (2 * len(intersection) / (len(set_A) + len(set_B)))

In [3]:
# 연산자로도 가능
intersection = set_A & set_B    # 교집합
union = set_A | set_B           # 합집합
difference = set_A - set_B      # 차집합

In [4]:
# 코사인 유사도의 한계
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 오타가 있을 경우 코사인 유사도는 0이 나옴
# apple vs apples -> ;완전히 다른 단어로 인식
corpus = [
    "apple",
    "apples"
]

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)

cos_sim = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])
print(f"코사인 유사도 : {cos_sim[0][0]:.3f}")
# -> 0.000 <- 완전히 다른 단어로 인식

코사인 유사도 : 0.000


In [5]:
# n-그램(Shingle)으로 오타 해결
text = 'monkey'
k = 2

# 문자열 슬라이싱으로 2-그램 생성
for i in range(len(text)-1):
    print(text[i : k+i])

# get_shingles: 단어를 n-그램 집합으로 만드는 함수
# set comprehension으로 중복 자동 제거
def get_shingles(text, k=2):
    return {text[i:k+i] for i in range(len(text)-1)} 

word_a, word_b = 'apple', 'apples'
shingle_a = get_shingles(word_a) 
shingle_b = get_shingles(word_b)
print(shingle_a)
print(shingle_b)

# 자카드 유사도 계산
inter = shingle_a.intersection(shingle_b)
uni = shingle_a.union(shingle_b)
sim = len(inter) / len(uni)
print(f"자카드 유사도 : {sim:.3f}")

mo
on
nk
ke
ey
{'pp', 'le', 'ap', 'pl'}
{'es', 'ap', 'pp', 'le', 'pl'}
자카드 유사도 : 0.800


## 006

In [6]:
from sklearn.datasets import fetch_20newsgroups

# fetch_20newsgroups : sklearn 내장 뉴스 그룹 데이터셋
# subset = 'train' : 학습 데이터만 가져오기
# subset = 'test' : 테스트 데이터만 가져오기
# categories : 사용할 토픽 선택
# remove = ('headers', 'footers', 'quotes') :
#   헤더(작성자 정보), 푸터(서명), 인용문 제거 -> 순수 본문만
# 20개토픽중에 선택 ( 무신론, 종교, 컴퓨터그래픽, 우주과학)
categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups_train = fetch_20newsgroups(
    subset='train',
    remove = ('headers', 'footers', 'quotes'),
    categories = categories
)
newsgroups_test = fetch_20newsgroups(
    subset='test',
    remove = ('headers', 'footers', 'quotes'),
    categories = categories
)

# 학습/테스트 데이터 크기 확인
len(newsgroups_train.data), len(newsgroups_test.data)

# target : 카테고리 번호 (0, 1, 2, 3)
# target_names : 카테고리 이름
print(newsgroups_train.data[100])
newsgroups_train.target[100]
newsgroups_train.target_names[newsgroups_train.target[100]]



This is a good point, but I think "average" people do not take up Christianity
so much out of fear or escapism, but, quite simply, as a way to improve their
social life, or to get more involved with American culture, if they are kids of
immigrants for example.  Since it is the overwhelming major religion in the
Western World (in some form or other), it is simply the choice people take if
they are bored and want to do something new with their lives, but not somethong
TOO new, or TOO out of the ordinary.  Seems a little weak, but as long as it
doesn't hurt anybody...
 

These are good quotes, and I agree with both of them, but let's make sure to
alter the scond one so that includes something like "...let him be, as long as
he is not preventing others from finding their peace." or something like that. 
(Of course, I suppose, if someone were REALLY "at peace", there would be no
need for inflicting evangelism)


Well, it is a sure thing we will have to live with them all our lives.  Their


'alt.atheism'

In [7]:
# 학습/테스트 데이터 분리
x_train = newsgroups_train.data
y_train = newsgroups_train.target
x_test = newsgroups_test.data
y_test = newsgroups_test.target

# CountVectorizer로 벡터화
# max_features=2000 : 빈도 높은 상위 2000개 단어만 사용
# fit_transform : 학습 데이터로 어휘 학습 + 변환
# transform : 테스트 데이터는 변환만 (fit 하면 안됨)
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2000)
x_train_cv = cv.fit_transform(x_train)
x_test_cv = cv.transform(x_test)

x_train_cv.shape, x_test_cv.shape

((2034, 2000), (1353, 2000))

In [8]:
from sklearn.naive_bayes import MultinomialNB

# multinomialNB : 나이브 베이즈 분류기
# 텍스트 분류에서 가장 많이 쓰는 모델 중 하나
# 각 단어가 독립적이라고 가정하고 확률로 분류
# 언어모델(MLE)이랑 원리가 같음
NB_clf = MultinomialNB()

# 모델 학습
NB_clf.fit(x_train_cv, y_train)

# score : 정확도 계산 (맞춘 개수 / 전체 개수)
# 학습 정확도, 테스트 정확도
print( NB_clf.score(x_train_cv, y_train) ), print( NB_clf.score(x_test_cv, y_test) )

# predict : 새로운 데이터 예측
# x_test_cv[:3] : 테스트 데이터 앞 3개 예측
NB_clf.predict(x_test_cv[:10]), y_test[:10]
# => 예측값과 실제값 비교

0.8200589970501475
0.7317073170731707


(array([2, 1, 1, 1, 1, 1, 0, 2, 0, 2]), array([2, 1, 1, 1, 1, 1, 2, 2, 0, 2]))

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# CountVectorizer 대신 TfidfVectorizer 사용
# 차이점 : 단순 빈도 대신 TF-IDF 가중치 사용
tfidf = TfidfVectorizer(max_features=2000)
x_train_tfidf = tfidf.fit_transform(x_train) 
x_test_tfidf = tfidf.transform(x_test)

#똑같이 나이브 베이즈로 학습
NB_clf = MultinomialNB()
NB_clf.fit(x_train_tfidf, y_train)

# .CountVectorizer vs TfidfVectorizer 정확도 비교
NB_clf.score(x_train_tfidf, y_train), NB_clf.score(x_test_tfidf, y_test)

(0.8525073746312685, 0.7376201034737621)

In [10]:
# 다른 모델들과 비교
from sklearn.linear_model import LogisticRegression, RidgeClassifier, LassoCV

# 3가지 모델 생성
logistic = LogisticRegression()
# 가장 기본적인 분류 모델 => 텍스트 분류에 성능 좋음
ridge = RidgeClassifier()
# 가중치 크기 제한 (L2 규제) => 과적합 방지
lasso = LassoCV()
# 불필요한 가중치 0으로 (L1규제) => 희소한 모델

# 각 모델 학습
logistic.fit(x_train_tfidf, y_train)
ridge.fit(x_train_tfidf, y_train)
lasso.fit(x_train_tfidf, y_train)

# 학습/테스트 정확도 비교
print(logistic.score(x_train_tfidf, y_train), logistic.score(x_test_tfidf, y_test))
print(ridge.score(x_train_tfidf, y_train), ridge.score(x_test_tfidf, y_test))
print(lasso.score(x_train_tfidf, y_train), lasso.score(x_test_tfidf, y_test))

0.9203539823008849 0.7317073170731707
0.9587020648967551 0.7427937915742794
0.48895982123641346 0.14656652054016017


In [11]:
# 모델이 학습 데이터에 너무 맞춰지면 → 과적합
# → 학습 정확도는 높은데 테스트 정확도는 낮음

# 규제 = 모델이 너무 복잡해지지 않도록 제한
# Ridge (L2): 가중치를 작게 유지
# Lasso (L1): 불필요한 가중치를 아예 0으로 만들기

In [12]:
import numpy as np
from sklearn.model_selection import GridSearchCV

# np.linspace(0.1, 10, 50) : 0.1부터 10까지 50개를 균등하게 나눈 리스트
# alpha : 규제 강도 (클수록 규제 강함 -> 단순한 모델)
alpha_lists = np.linspace(0.1, 10, 50)

# GridSearchCV : 여러 파라미터 조합을 자동으로 시도해서 최적값 찾기
# param_grid : 시도할 파라미터 딕셔너리
# 내부적으로 교차검증(Cross Validation)도 함께 수행
params = {
    'alpha' : alpha_lists
}

grid = GridSearchCV(RidgeClassifier(), param_grid=params)
grid.fit(x_train_tfidf, y_train)

# best_estimator_ : 가장 좋은 성능의 모델
best_model = grid.best_estimator_
best_model.score(x_train_tfidf, y_train), best_model.score(x_test_tfidf, y_test)

(0.9360865290068829, 0.7479674796747967)

In [13]:
# 한국어 뉴스 분류
news_data = {
    'content': [
        # IT/과학
        "삼성전자가 차세대 폴더블 스마트폰을 공개하며 시장 공략에 나섰습니다.",
        "인공지능 기술이 발전함에 따라 반도체 수요가 급증하고 있습니다.",
        "구글의 새로운 AI 모델이 인간과의 대화에서 자연스러운 반응을 보였습니다.",
        # 경제
        "미국 연준이 금리를 동결하면서 국내 증시는 혼조세를 보였습니다.",
        "최근 물가 상승률이 둔화되면서 하반기 경기 회복 기대감이 커지고 있습니다.",
        "대기업들의 실적 발표가 이어지는 가운데 반도체 업종이 강세를 보였습니다.",
        # 사회
        "이번 주말 서울 도심에서 대규모 집회가 예정되어 교통 혼잡이 예상됩니다.",
        "경찰은 최근 급증하는 보이스피싱 범죄를 막기 위해 집중 단속에 나섰습니다.",
        "정부는 저출산 문제 해결을 위해 새로운 육아 지원 정책을 발표했습니다.",
        # 정치
        "여야 정치권은 국회 본회의를 열고 민생 법안 처리에 합의했습니다.",
        "대통령은 국무회의에서 경제 활성화를 위한 규제 개혁을 강조했습니다.",
        "새로운 정당 창당 소식에 정치권의 판도가 요동치고 있습니다."
    ],
    'category': ['IT/과학', 'IT/과학', 'IT/과학', '경제', '경제', '경제', '사회', '사회', '사회', '정치', '정치', '정치']
}

# 한국어 토크나이저 함수
# TfidfVectorizer에 tokenizer 파라미터로 넣을 함수
# 명사만 추출 + 2글자 이상만
from konlpy.tag import Okt
okt = Okt()

def korean_tokenizer(text):
    return [word for word in okt.nouns(text) if len(word) >= 2]

# tokenizer = korean_tokenizer : 직접 만든 토크나이저 사용
# 내부적으로 토큰화를 korean_tokenizer 함수로 수행
tfidf = TfidfVectorizer(tokenizer=korean_tokenizer)

x, y = news_data['content'], news_data['category']
x_tfidf = tfidf.fit_transform(x)

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_tfidf, y, random_state=42)

# alpha = 0.1 : 규제 약하게 (데이터가 적으니까)
nb = MultinomialNB(alpha=0.1)
nb.fit(x_train, y_train)
print( nb.score(x_train, y_train) ), print( nb.score(x_test, y_test) )

# 예측 결과 확인
nb.predict(x_test), y_test

c:\Users\Playdata\miniconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1.0
0.6666666666666666


(array(['IT/과학', '정치', 'IT/과학'], dtype='<U5'), ['정치', '정치', 'IT/과학'])

In [14]:
# 기본: 공백 기준 토큰화
tfidf = TfidfVectorizer()

# 커스텀: 내가 만든 함수로 토큰화
tfidf = TfidfVectorizer(tokenizer=korean_tokenizer)

## 007

In [15]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

categories = ['alt.atheism', 'talk.religion.misc', 'comp.graphics', 'sci.space']
newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

# max_features=1000
# min_df = 5 : 최소 5개 문서에 등장한 단어만
#               -> 너무 희귀한 단어 제거
# max_df = 0.5 : 전체 문서의 50% 이상 등장한 단어 제거
#               -> 너무 흔한 단어 제거 (불용어 효과)
# stop_words = 'english' : 영어 불용어 제거
tfidf = TfidfVectorizer(
    max_features=1000,
    min_df=5,
    max_df=0.5,
    stop_words='english'
)
X_tfidf = tfidf.fit_transform(newsgroups.data)
print(f"원본 TF-IDF 행렬 차원 : {X_tfidf.shape}")


원본 TF-IDF 행렬 차원 : (2034, 1000)


In [16]:
# TruncatedSVD로 차원 축소
from sklearn.decomposition import TruncatedSVD

# TruncatedSVD : LSA 구현체
# n_components = 10 : 1000차원 -> 100 차원으로 축소
#                     100개의 잠재 의미(토픽)만 남김
svd = TruncatedSVD(n_components = 100, random_state=42)

# X_tfidf : (2034, 1000) 입력
# x_svd : (2034, 100) 출력 <- 차원 감소
x_svd = svd.fit_transform(X_tfidf)

# explained_variance_ratio_ : 각 토픽이 전체 데이터를 얼마나 설명하는지
# sum() : 100개 토픽이 전체 데이터의 몇 %를 설명하는지
x_svd.shape, svd.explained_variance_ratio_.sum()
# -> (2043, 100), 0.35 => 100개 토픽이 전체의 35% 설명

((2034, 100), np.float64(0.3507431261763808))

In [17]:
import numpy as np
# tfidf.get_feature_names_out() : 어휘 목록(1000개 단어)
terms = tfidf.get_feature_names_out()

#svd.components_: 각 토픽별 단어 가중치 행렬
# shape: (10, 1000) -> 10개 토픽 X 1000개 단어
for i, comp in enumerate(svd.components_):
    # np.argsort(-comp)[:10] : 가중치 높은 상위 100개 단어 인덱스
    # -comp: 내림차순 정렬을 위해 음수로 변환
    top_terms_index = np.argsort(-comp)[:100]
    print(top_terms_index)

    # 인덱스로 실제 단어 가져오기
    top_terms = [ terms[idx] for idx in top_terms_index ]
    print(f"topic {i+1}: {', '.join(top_terms)}")

[275 639 461 372 897 833 496 469 272 376 903 775 947 968 933 123 256 526
 896 768 452 722 893 756 587 962 659 380 578 687 126 494 508 985 718 733
 287 913 592 705 125 895 996 934 505 375 502 871 130 257 377 665 171  48
 403 338 681 273 940 137 720 982 420 283 261 542 305 915 510 515 541 220
 678 997 449 320 823 899 992 230 491 417 337 570 569 885 328 708 481 186
 522 152 511 663 185 308 104 723 112 602]
topic 1: don, people, just, god, think, space, like, know, does, good, time, say, ve, way, use, believe, did, make, things, said, jesus, really, thanks, right, need, want, point, graphics, nasa, program, bible, life, long, world, read, religion, edu, true, new, question, better, thing, years, used, ll, going, little, sure, bit, didn, got, post, christian, actually, help, files, problem, doesn, using, book, real, work, image, earth, different, mean, evidence, try, look, lot, maybe, course, probably, yes, isn, fact, software, thought, wrong, data, let, idea, file, moral, moon, tell, far, 

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# x_svd[0:1] : 첫 번째 문서 벡터 (1 X 100)
# x_svd : 전체 문서  벡터 (2034 X 10)
# cosine_similarity : 첫 번째 문서와 전체 문서의 유사도
sim_scores = cosine_similarity(x_svd[0:1], x_svd)

# 유사도 높은 순으로 상위 5개 인덱스
top_index = np.argsort(-sim_scores)[:5]
print(f"가장 유사한 문서 top5 index : {top_index}")
print(f"유사도 점수 : {sim_scores[0][top_index]}")

# 첫 번째 문서 내용과 카테고리 확인
print( newsgroups.data[0] ), print( newsgroups.target_names[newsgroups.target[0]] )

# 전체 카테고리 목록 확인
newsgroups.target_names


가장 유사한 문서 top5 index : [[  0 651 790 ... 871 238 622]]
유사도 점수 : [[ 1.          0.78961842  0.77159879 ... -0.0968836  -0.09764711
  -0.09845958]]
Hi,

I've noticed that if you only save a model (with all your mapping planes
positioned carefully) to a .3DS file that when you reload it after restarting
3DS, they are given a default position and orientation.  But if you save
to a .PRJ file their positions/orientation are preserved.  Does anyone
know why this information is not stored in the .3DS file?  Nothing is
explicitly said in the manual about saving texture rules in the .PRJ file. 
I'd like to be able to read the texture rule information, does anyone have 
the format for the .PRJ file?

Is the .CEL file format available from somewhere?

Rych
comp.graphics


['alt.atheism', 'comp.graphics', 'sci.space', 'talk.religion.misc']

In [22]:
import re
from konlpy.tag import Okt, Kkma
import pandas as pd

# 정규 표현식 연습
raw_text = "안녕하세요!!! 123반갑습니다... @@@자연어 처리는 재미있어요 ^_^"

cleaned_text = re.sub(r'[^가-힣\s]', '', raw_text)
print(cleaned_text)

안녕하세요 반갑습니다 자연어 처리는 재미있어요 


In [23]:
okt = Okt()
kkma = Kkma()

text = '나는 사과를 먹는다'

# okt.pos() : (단어, 품사) 튜플 리스트 반환
print(okt.pos(text))
print(kkma.pos(text))

[('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb')]
[('나', 'VV'), ('는', 'ETD'), ('사과', 'NNG'), ('를', 'JKO'), ('먹', 'VV'), ('는', 'EPT'), ('다', 'EFN')]


In [24]:
text = '사과가 맛있다, 나는 사과를 먹는다'

print(okt.pos(text))

# 품사로 불용어 제거
# 단어 자체가 아닌 품사 기준은로 제거
# Josa(조사), Punctuation(구두점) 제거
stopwords = ['Josa', 'Punctuation']
cleaned_text = [t for t, pos in okt.pos(text) if pos not in stopwords]
print(cleaned_text)

[('사과', 'Noun'), ('가', 'Josa'), ('맛있다', 'Adjective'), (',', 'Punctuation'), ('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb')]
['사과', '맛있다', '나', '사과', '먹는다']


In [29]:
import pandas as pd

df = pd.read_csv('daum_movie_review.csv')

# 'review' 컬럼만 리스트로 추출
text = list(df['review'])

# 한글과 공백만 남기기
cleaned_text = [ re.sub(r'[^가-힣\s]', '',  doc) for doc in text ]

# 품사 필터링 (명사, 동사, 형용사만 추출)
POS = ['Noun', 'Verb', 'Adjective']
okt = Okt()

# 각 문서에서 원하는 품사만 추출
tokenized = []
for doc in cleaned_text:
    tokens = [t for t, pos in okt.pos(doc) if pos in POS]
    tokenized.append(tokens)

# 전체 토큰을 하나의 리스트로 합치기
all_tokens = [token for tokens in tokenized for token in tokens]

# TTR 계산
V = list(set(token for tokens in tokenized for token in tokens))
TTR = len(V) / len(all_tokens)
print(f"TTR : {TTR:.4f}")

TTR : 0.1377


In [30]:
import numpy as np

# 토큰 수 기준 상위 100개 리뷰 추출
# -len(doc) : 음수로 바꿔서 argsort -> 내림차순 효과
top100_index = np.argsort([-len(doc) for doc in tokenized])[:100]

# 인덱스로 실제 리뷰 가졍오기
top100_corpus = [tokenized[index] for index in top100_index]

# 첫 번째 리뷰의 TTR 계산
# set(top100_corpus[0]) : 고유 토큰
# len(top100_corpus[0]) : 전체 토큰
ttr = len(set(top100_corpus[0])) / len(top100_corpus[0])
print(ttr)

0.823076923076923


In [32]:
import pandas as pd
import re
from konlpy.tag import Okt

daum_df = pd.read_csv('daum_movie_review.csv')

# to_numpy() : DataFrame 컬럼 -> numpy 배열로 변환
# list()와 차이 : numpy배열은 수학 연산 가능
corpus = daum_df['review'].to_numpy()

cleaned_corpus = [re.sub(r'[^가-힣\s]', '', doc) for doc in corpus]
# 전처리 함수화
def pos_preprocess(doc):
    okt = Okt()
    temp = []
    for token, pos in okt.pos(doc):
        if pos in ['Noun', 'Verb', 'Adjective'] and len(token) >= 2:
            temp.append(token)
    return temp

# 빈 문서 방지
# pos_preprocess("...") : 특수 문자만 있으면 [] 반환
# len(temp) > 0 : 빈 리스트는 제외
cleaned_corpus_pos = []
for doc in cleaned_corpus:
    temp = pos_preprocess(doc)
    if len(temp) > 0:
        cleaned_corpus_pos.append(temp)


In [33]:
# TTR 계산 함수
def calc_ttr(text):
    tokens = okt.morphs(text) # 형태소 분리
    if not tokens: return 0 # 빈 문서 방지
    return len(set(tokens)) / len(tokens)

# 스팸 리뷰 vs 정상 리뷰 TTR 비교
spam_review = '추천합니다 추천합니다 추천합니다 추천합니다 추천합니다'
normal_review = '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.'

calc_ttr(spam_review), calc_ttr(normal_review)


(0.2, 0.8571428571428571)

In [ ]:
import re
import numpy as np
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from collections import Counter

okt = Okt()

# ================================================
# 0. 데이터 로드
# ================================================
df = pd.read_csv('daum_movie_review.csv')
corpus = df['review'].to_numpy()
labels = df['rating'].to_numpy()

# ================================================
# 1. 전처리 (Preprocessing)
# ================================================
# 1-1. 한글과 공백만 추출
cleaned_corpus = [re.sub(r'[^가-힣\s]', '', doc) for doc in corpus]

# 1-2. 형태소 분석 + 품사 필터링 함수
def pos_filter(doc):
    tokens = [
        token for token, pos in okt.pos(doc)
        if pos in ['Noun', 'Verb', 'Adjective']
        and len(token) >= 2
    ]
    return tokens

# 1-3. 전처리 적용 (빈 문서 제외)
tokenized = []
for doc in cleaned_corpus:
    tokens = pos_filter(doc)
    if len(tokens) > 0:
        tokenized.append(tokens)

print("=== 전처리 결과 (상위 3개) ===")
for i, doc in enumerate(tokenized[:3]):
    print(f"d{i+1}: {doc}")

# ================================================
# 2. 기본 통계 (Basic Statistics)
# ================================================
N = len(tokenized)
all_tokens = [token for doc in tokenized for token in doc]
T = len(all_tokens)
V = list(set(all_tokens))
TTR = len(V) / T

freq = Counter(all_tokens)

non_zero = sum(len(set(doc)) for doc in tokenized)
sparsity = 1 - (non_zero / (N * len(V)))

print(f"\n=== 기본 통계 ===")
print(f"문서 수 N        : {N}")
print(f"전체 토큰 수 T   : {T}")
print(f"어휘 크기 |V|    : {len(V)}")
print(f"TTR              : {TTR:.4f}")
print(f"희소성           : {sparsity:.4f}")
print(f"상위 5개 단어    : {freq.most_common(5)}")

# TTR 기반 스팸 탐지
def calc_ttr(text):
    tokens = okt.morphs(text)
    if not tokens: return 0
    return len(set(tokens)) / len(tokens)

print(f"\n=== TTR 스팸 탐지 ===")
for doc in corpus[:5]:
    ttr = calc_ttr(doc)
    label = "스팸 의심 ❌" if ttr < 0.4 else "정상 ✅"
    print(f"TTR: {ttr:.3f} | {label} | {doc[:30]}...")

# ================================================
# 3. 집합 기반 유사도 (Set-based Similarity)
# ================================================
def jaccard_similarity(doc_a, doc_b):
    set_a, set_b = set(doc_a), set(doc_b)
    if not set_a | set_b: return 0
    return len(set_a & set_b) / len(set_a | set_b)

def dice_similarity(doc_a, doc_b):
    set_a, set_b = set(doc_a), set(doc_b)
    if not set_a and set_b: return 0
    return 2 * len(set_a & set_b) / (len(set_a) + len(set_b))

print("\n=== 집합 기반 유사도 (상위 3개 문서) ===")
for i in range(3):
    for j in range(i+1, 3):
        jaccard = jaccard_similarity(tokenized[i], tokenized[j])
        dice = dice_similarity(tokenized[i], tokenized[j])
        print(f"d{i+1}-d{j+1} | 자카드: {jaccard:.3f} | 다이스: {dice:.3f}")

# ================================================
# 4. 벡터화 (Vectorization)
# ================================================
# 커스텀 토크나이저
def korean_tokenizer(text):
    return [token for token, pos in okt.pos(text)
            if pos in ['Noun', 'Verb', 'Adjective']
            and len(token) >= 2]

tfidf = TfidfVectorizer(
    tokenizer=korean_tokenizer,
    max_features=1000,
    min_df=2,
    max_df=0.9
)
corpus_joined = [" ".join(doc) for doc in tokenized]
X = tfidf.fit_transform(corpus_joined)
print(f"\n=== 벡터화 결과 ===")
print(f"TF-IDF 행렬 크기: {X.shape}")

# ================================================
# 5. 차원 축소 (Dimensionality Reduction)
# ================================================
svd = TruncatedSVD(n_components=10, random_state=42)
X_svd = svd.fit_transform(X)
print(f"\n=== 차원 축소 결과 ===")
print(f"축소 후 크기  : {X_svd.shape}")
print(f"설명력        : {svd.explained_variance_ratio_.sum():.4f}")

terms = tfidf.get_feature_names_out()
for i, comp in enumerate(svd.components_):
    top_terms = [terms[idx] for idx in np.argsort(-comp)[:5]]
    print(f"topic {i+1}: {', '.join(top_terms)}")

# ================================================
# 6. 코사인 유사도 (Cosine Similarity)
# ================================================
sim_scores = cosine_similarity(X_svd[0:1], X_svd)
top_index = np.argsort(-sim_scores).reshape(-1)[:5]
print(f"\n=== 유사 문서 Top5 ===")
for rank, idx in enumerate(top_index):
    print(f"{rank+1}위 | 유사도: {sim_scores[0][idx]:.4f} | {corpus[idx][:30]}...")

# ================================================
# 7. 모델 학습 및 평가 (Model Training)
# ================================================
x_train, x_test, y_train, y_test = train_test_split(
    X, labels[:len(tokenized)], random_state=42, test_size=0.2
)

nb = MultinomialNB(alpha=0.1)
nb.fit(x_train, y_train)

params = {'alpha': np.linspace(0.1, 10, 50)}
grid = GridSearchCV(MultinomialNB(), param_grid=params)
grid.fit(x_train, y_train)
best_model = grid.best_estimator_

print(f"\n=== 모델 평가 ===")
print(f"NB 학습 정확도   : {nb.score(x_train, y_train):.4f}")
print(f"NB 테스트 정확도 : {nb.score(x_test, y_test):.4f}")
print(f"최적 모델 정확도 : {best_model.score(x_test, y_test):.4f}")